# Telecom Network Analysis — Análise Exploratória

**Dataset:** Telstra Network Disruptions (Kaggle, 2015)
**Objetivo:** entender a estrutura, a qualidade e os limites do dado **antes** de
construir qualquer análise ou dashboard.

Uma equipe de Network Operations recebeu dados históricos de incidentes de rede e
precisa identificar padrões de falha, localidades problemáticas e recursos críticos.
Este notebook responde uma pergunta anterior a todas essas: **o que estes dados
conseguem, e o que não conseguem, responder?**

> **Sobre gráficos:** este notebook é deliberadamente tabular. O GitHub não renderiza
> saídas de Plotly no preview de `.ipynb`, então as visualizações vivem em
> `reports/figures/` e no dashboard Streamlit, onde de fato aparecem.

In [1]:
import sys
from pathlib import Path

# Permite importar `src` ao rodar o notebook a partir de notebooks/
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd

from src.data_loader import load_raw_data
from src.config import FAULT_SEVERITY_LABELS

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 100)

raw = load_raw_data()
print("Tabelas carregadas:", ", ".join(raw.as_dict()))

Tabelas carregadas: train, severity_type, event_type, resource_type, log_feature


---

## 1. Visão geral dos arquivos

Primeira pergunta de qualquer exploração: **o que existe, e de que tamanho?**

In [2]:
visao = pd.DataFrame([
    {
        "tabela": nome,
        "linhas": len(df),
        "colunas": df.shape[1],
        "ids_distintos": df["id"].nunique(),
        "memoria_MB": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    }
    for nome, df in raw.as_dict().items()
])
visao["relacao_com_incidente"] = [
    "1:1 (chave primária)" if r.linhas == r.ids_distintos else "1:N"
    for r in visao.itertuples()
]
visao

,tabela,linhas,colunas,ids_distintos,memoria_MB,relacao_com_incidente
0,train,7381,3,7381,0.20,1:1 (chave primária)
1,severity_type,18552,2,18552,0.55,1:1 (chave primária)
2,event_type,31170,2,18552,0.86,1:N
3,resource_type,21076,2,18552,0.62,1:N
4,log_feature,58671,3,18552,1.95,1:N


`train` e `severity_type` têm uma linha por incidente. As outras três têm
**várias linhas por incidente** — é isso que caracteriza uma relação **1:N** (um para
muitos) e o que vai justificar tabelas separadas no banco em vez de uma planilha larga.

In [3]:
for nome, df in raw.as_dict().items():
    print(f"--- {nome} ---")
    print(df.dtypes.to_string(), "\n")

--- train ---
id                 int64
location          string
fault_severity      int8 

--- severity_type ---
id                int64
severity_type    string 

--- event_type ---
id             int64
event_type    string 

--- resource_type ---
id                int64
resource_type    string 

--- log_feature ---
id              int64
log_feature    string
volume          int64 



Os tipos foram **declarados** em `src/data_loader.py`, não inferidos pelo pandas.
Se um `id` viesse corrompido, a inferência silenciosamente transformaria a coluna em
`object` e todos os JOINs a jusante falhariam de forma difícil de rastrear.

---

## 2. Qualidade dos dados

As três verificações que abrem qualquer projeto: **nulos, duplicatas e integridade
referencial**.

In [4]:
qualidade = pd.DataFrame([
    {
        "tabela": nome,
        "valores_nulos": int(df.isna().sum().sum()),
        "linhas_duplicadas": int(df.duplicated().sum()),
    }
    for nome, df in raw.as_dict().items()
])
qualidade

,tabela,valores_nulos,linhas_duplicadas
0,train,0,0
1,severity_type,0,0
2,event_type,0,0
3,resource_type,0,0
4,log_feature,0,0


**Zero nulos e zero duplicatas em todo o dataset.**

Isso é incomum e vale registrar com honestidade: este é um dado já tratado pela
Telstra antes da publicação. A consequência para o projeto é que o módulo de
*cleaning* será pequeno — inventar tratamento de valores ausentes onde não há
valores ausentes seria teatro, não análise.

In [5]:
# Integridade referencial: todo incidente rotulado aparece nas tabelas satélite?
train_ids = set(raw.train["id"])

integridade = pd.DataFrame([
    {
        "tabela": nome,
        "incidentes_de_train_presentes": len(train_ids & set(df["id"])),
        "incidentes_de_train_ausentes": len(train_ids - set(df["id"])),
    }
    for nome, df in raw.as_dict().items() if nome != "train"
])
integridade

,tabela,incidentes_de_train_presentes,incidentes_de_train_ausentes
0,severity_type,7381,0
1,event_type,7381,0
2,resource_type,7381,0
3,log_feature,7381,0


Nenhum órfão. Todo incidente de `train` tem correspondência nas quatro tabelas
satélite, o que significa que os JOINs não vão perder linhas silenciosamente.

---

## 3. Recorte do universo de análise

`fault_severity` — a variável que queremos explicar — **só existe em `train.csv`**.
Os 11.171 incidentes de `test.csv` eram o conjunto oculto de avaliação da competição
Kaggle e nunca tiveram o gabarito publicado.

Como todas as perguntas de negócio deste projeto envolvem gravidade, o universo de
análise são os **7.381 incidentes rotulados**.

In [6]:
ev  = raw.event_type[raw.event_type["id"].isin(train_ids)]
res = raw.resource_type[raw.resource_type["id"].isin(train_ids)]
log = raw.log_feature[raw.log_feature["id"].isin(train_ids)]
sev = raw.severity_type[raw.severity_type["id"].isin(train_ids)]

recorte = pd.DataFrame([
    {"tabela": n, "linhas_apos_recorte": len(d), "incidentes_cobertos": d["id"].nunique()}
    for n, d in [("event_type", ev), ("resource_type", res),
                 ("log_feature", log), ("severity_type", sev)]
])
recorte

,tabela,linhas_apos_recorte,incidentes_cobertos
0,event_type,12468,7381
1,resource_type,8460,7381
2,log_feature,23851,7381
3,severity_type,7381,7381


---

## 4. A variável-alvo: `fault_severity`

**Variável-alvo** (*target*) é o que queremos explicar. Aqui, quantas falhas o
incidente efetivamente causou.

In [7]:
alvo = raw.train["fault_severity"].value_counts().sort_index().to_frame("incidentes")
alvo["%"] = (alvo["incidentes"] / len(raw.train) * 100).round(2)
alvo["significado"] = alvo.index.map(FAULT_SEVERITY_LABELS)
alvo

,incidentes,%,significado
fault_severity,,,
0,4784,64.82,Sem falha reportada
1,1871,25.35,Poucas falhas
2,726,9.84,Muitas falhas


**A distribuição é desbalanceada:** quase dois terços dos incidentes não geraram
falha, e apenas ~10% foram graves.

Isso tem uma consequência prática que atravessa o projeto inteiro: **ranking por
contagem absoluta de incidentes graves vai apenas reproduzir o ranking de volume
total**. Uma localidade grande gera muitos incidentes de tudo, inclusive graves.
Para responder "onde a rede está pior?" é preciso usar **taxa**, não contagem —
e isso traz o problema do denominador pequeno, tratado na seção 6.

---

## 5. A armadilha central: `severity_type` não é `fault_severity`

Duas colunas com nomes parecidos e significados completamente diferentes:

| | `fault_severity` | `severity_type` |
|---|---|---|
| O que é | **resultado** do incidente | tipo da mensagem de alerta do log |
| Papel | variável-alvo (*target*) | atributo preditor (*feature*) |
| Valores | 0, 1, 2 | 5 categorias |
| Disponível para | só `train` (7.381) | todos (18.552) |

Reportar a distribuição de `severity_type` como se fosse "distribuição de gravidade"
produziria números que somam 100%, gráficos bonitos e conclusões inteiramente falsas.

Como `severity_type` é *feature* e `fault_severity` é *target*, a pergunta legítima é:
**uma explica a outra?**

In [8]:
m = raw.train.merge(sev, on="id")

ct = pd.crosstab(m["severity_type"], m["fault_severity"])
ct.columns = [f"fault_{c}" for c in ct.columns]
ct["total"] = ct.sum(axis=1)
ct["%_graves"] = (ct["fault_2"] / ct["total"] * 100).round(1)
ct.sort_values("total", ascending=False)

,fault_0,fault_1,fault_2,total,%_graves
severity_type,,,,,
severity_type 2,2652,693,246,3591,6.9
severity_type 1,1778,1117,480,3375,14.2
severity_type 4,338,50,0,388,0.0
severity_type 5,12,11,0,23,0.0
severity_type 3,4,0,0,4,0.0


**Achado relevante.** A taxa média de incidentes graves no dataset é **9,84%**.

- `severity_type 1` → **14,2%** de graves (acima da média)
- `severity_type 2` → **6,9%** de graves (abaixo da média)
- `severity_type 3`, `4` e `5` → **0 incidentes graves**

O caso do `severity_type 4` é o mais forte: são **388 incidentes e nenhum grave**.
Se a taxa base de 9,84% valesse ali, esperaríamos cerca de 38. Zero em 388 não é
coincidência de amostra pequena — é sinal.

Operacionalmente: o tipo de alarme emitido pelo log **carrega informação real** sobre
a gravidade do que vem a seguir. Alarmes tipo 1 merecem prioridade maior que tipo 2,
e os tipos 3/4/5 aparentemente nunca escalam.

---

## 6. Localidades: cauda longa e o problema do denominador

**Cardinalidade** = quantos valores distintos uma coluna tem. Cardinalidade alta em
uma variável categórica muda como ela pode ser analisada e modelada.

In [9]:
cardinalidade = pd.DataFrame([
    {"coluna": "location",      "valores_distintos": raw.train["location"].nunique()},
    {"coluna": "event_type",    "valores_distintos": ev["event_type"].nunique()},
    {"coluna": "resource_type", "valores_distintos": res["resource_type"].nunique()},
    {"coluna": "severity_type", "valores_distintos": sev["severity_type"].nunique()},
    {"coluna": "log_feature",   "valores_distintos": log["log_feature"].nunique()},
])
cardinalidade

,coluna,valores_distintos
0,location,929
1,event_type,49
2,resource_type,10
3,severity_type,5
4,log_feature,331


In [10]:
# Quantos incidentes cada localidade tem?
por_loc = raw.train["location"].value_counts()

print(f"Localidades distintas: {len(por_loc)}")
print(f"Incidentes por localidade -> min={por_loc.min()}  mediana={por_loc.median():.0f}  "
      f"média={por_loc.mean():.2f}  max={por_loc.max()}")
print()
print("Distribuição do tamanho da amostra por localidade:")
faixas = pd.cut(por_loc, bins=[0, 1, 5, 10, 20, 50, 100],
                labels=["1 incidente", "2-5", "6-10", "11-20", "21-50", "51+"])
faixas.value_counts().sort_index().to_frame("qtd_localidades")

Localidades distintas: 929
Incidentes por localidade -> min=1  mediana=3  média=7.95  max=85

Distribuição do tamanho da amostra por localidade:


,qtd_localidades
count,
1 incidente,241
2-5,338
6-10,149
11-20,107
21-50,75
51+,19


**241 localidades aparecem uma única vez.** Isso é o que se chama de
**cauda longa** (*long tail*): pouquíssimas localidades concentram muitos incidentes e
centenas aparecem uma ou duas vezes.

Agora o problema concreto — o ranking de "pior localidade" por taxa de gravidade:

In [11]:
g = (raw.train
     .assign(grave=(raw.train["fault_severity"] == 2).astype(int))
     .groupby("location", observed=True)
     .agg(incidentes=("id", "count"), graves=("grave", "sum")))
g["taxa_%"] = (g["graves"] / g["incidentes"] * 100).round(1)

print("RANKING SEM CORTE MÍNIMO:")
g.sort_values(["taxa_%", "incidentes"], ascending=[False, False]).head(8)

RANKING SEM CORTE MÍNIMO:


,incidentes,graves,taxa_%
location,,,
location 866,3,3,100.0
location 674,2,2,100.0
location 867,2,2,100.0
location 926,2,2,100.0
location 531,1,1,100.0
location 540,1,1,100.0
location 551,1,1,100.0
location 555,1,1,100.0


In [12]:
print(f"Localidades com taxa de 100%: {(g['taxa_%'] == 100).sum()}")
print(f"  destas, com apenas 1 incidente:  {((g['taxa_%'] == 100) & (g['incidentes'] == 1)).sum()}")
print(f"  destas, com 5 incidentes ou mais: {((g['taxa_%'] == 100) & (g['incidentes'] >= 5)).sum()}")

Localidades com taxa de 100%: 14
  destas, com apenas 1 incidente:  10
  destas, com 5 incidentes ou mais: 0


**Taxa** é uma divisão: `graves ÷ total`. Quando o **denominador** é minúsculo, a
taxa só consegue assumir valores extremos — com 1 incidente, os únicos resultados
possíveis são 0% ou 100%. A taxa não está medindo a qualidade da rede naquela
localidade; está medindo que houve uma única observação. Isso se chama
**ruído amostral** (*sampling noise*).

Com **929 localidades** sendo avaliadas ao mesmo tempo, é praticamente garantido que
algumas tenham azar puro — o **problema das comparações múltiplas**.

A correção é exigir um **corte mínimo** (*threshold*) de incidentes:

In [13]:
CORTE = 20  # decisão analítica: precisa ser declarada, não escondida

f = g[g["incidentes"] >= CORTE].sort_values("taxa_%", ascending=False)
print(f"Localidades com pelo menos {CORTE} incidentes: {len(f)} de {len(g)}")
print(f"Taxa média de graves no dataset inteiro: "
      f"{(raw.train['fault_severity'] == 2).mean()*100:.2f}%\n")
f.head(8)

Localidades com pelo menos 20 incidentes: 102 de 929
Taxa média de graves no dataset inteiro: 9.84%



,incidentes,graves,taxa_%
location,,,
location 1100,45,33,73.3
location 995,40,22,55.0
location 1086,33,16,48.5
location 600,64,29,45.3
location 962,45,20,44.4
location 638,25,11,44.0
location 1075,28,12,42.9
location 1107,78,33,42.3


Agora o ranking é acionável. `location 1100` tem **33 incidentes graves em 45** —
mais de 7 vezes a taxa média, sustentado por amostra suficiente. Isso é sinal, não ruído.

Vale notar: a localidade com **mais incidentes no total** não é a de maior taxa.
Volume e gravidade são fenômenos distintos.

In [14]:
# Volume total x taxa de gravidade: as duas visões não coincidem
top_volume = g.sort_values("incidentes", ascending=False).head(5)[["incidentes", "graves", "taxa_%"]]
print("TOP 5 POR VOLUME DE INCIDENTES:")
top_volume

TOP 5 POR VOLUME DE INCIDENTES:


,incidentes,graves,taxa_%
location,,,
location 821,85,28,32.9
location 1107,78,33,42.3
location 734,75,30,40.0
location 126,71,0,0.0
location 1008,71,16,22.5


---

## 7. Eventos e recursos: concentração

Antes de analisar categorias, vale saber se elas estão **concentradas** (poucas
dominam) ou **espalhadas**. Isso determina se um gráfico de barras do "top 10" conta
a história ou esconde a maior parte dela.

In [15]:
concentracao = []
for nome, df, col in [("event_type", ev, "event_type"),
                      ("resource_type", res, "resource_type"),
                      ("log_feature", log, "log_feature")]:
    vc = df[col].value_counts(normalize=True)
    concentracao.append({
        "tabela": nome,
        "categorias": df[col].nunique(),
        "top3_cobre_%": round(vc.head(3).sum() * 100, 1),
        "top10_cobre_%": round(vc.head(10).sum() * 100, 1),
    })
pd.DataFrame(concentracao)

,tabela,categorias,top3_cobre_%,top10_cobre_%
0,event_type,49,65.5,92.4
1,resource_type,10,93.2,100.0
2,log_feature,331,23.0,45.4


Leitura:

- **`resource_type`** é extremamente concentrado — 3 categorias cobrem 93% e o top 10
  cobre 100% (só existem 10). Um gráfico de barras conta a história inteira.
- **`event_type`** também concentra: top 10 cobre 92% das 49 categorias.
- **`log_feature`** é o oposto — 331 categorias e o top 10 cobre apenas 45%. Um "top 10"
  aqui **esconde mais da metade dos dados**. Esta variável precisa ser tratada por
  agregação (soma de volume por incidente), não por ranking de categorias.

In [16]:
print("TOP 10 event_type:")
ev["event_type"].value_counts().head(10).to_frame("incidentes")

TOP 10 event_type:


,incidentes
event_type,
event_type 11,3068
event_type 35,2693
event_type 34,2411
event_type 15,1724
event_type 20,557
event_type 54,264
event_type 13,247
event_type 23,198
event_type 42,185


In [17]:
print("resource_type (todas as 10 categorias):")
res["resource_type"].value_counts().to_frame("incidentes")

resource_type (todas as 10 categorias):


,incidentes
resource_type,
resource_type 8,4051
resource_type 2,3585
resource_type 6,247
resource_type 7,225
resource_type 4,144
resource_type 9,77
resource_type 3,58
resource_type 10,35
resource_type 1,34


---

## 8. `log_feature` e `volume`: forte assimetria

`volume` é a única variável **numérica contínua** do dataset.

In [18]:
v = log["volume"]
pd.DataFrame([{
    "min": v.min(), "q25": v.quantile(.25), "mediana": v.median(),
    "q75": v.quantile(.75), "p95": v.quantile(.95), "p99": v.quantile(.99),
    "max": v.max(), "média": round(v.mean(), 2), "assimetria (skew)": round(v.skew(), 2),
}])

,min,q25,mediana,q75,p95,p99,max,média,assimetria (skew)
0,1,1.0,2.0,7.0,42.0,114.5,877,9.85,10.05


In [19]:
print(f"% de linhas com volume <= 10: {(v <= 10).mean()*100:.1f}%")
print(f"Média ({v.mean():.2f}) é {v.mean()/v.median():.1f}x a mediana ({v.median():.0f})")

% de linhas com volume <= 10: 80.9%
Média (9.85) é 4.9x a mediana (2)


**Assimetria** (*skewness*) mede o quanto uma distribuição pende para um lado.
Valores acima de 1 já indicam forte assimetria; aqui é **10,05**.

Sintoma clássico: a **média (9,85) é quase 5 vezes a mediana (2)**. Quando isso
acontece, a média deixa de representar o caso típico — ela está sendo puxada por uma
minoria de valores enormes (o máximo é 877, contra um p95 de 42). 81% das linhas têm
volume ≤ 10.

Consequências práticas:
- Usar **mediana**, não média, para descrever o volume típico.
- Gráficos de volume precisam de **escala logarítmica**, senão 99% das barras ficam
  coladas no zero e só o outlier aparece.

---

## 9. Estrutura 1:N — quantos registros por incidente

In [20]:
fanout = []
for nome, df in [("event_type", ev), ("resource_type", res), ("log_feature", log)]:
    c = df.groupby("id", observed=True).size()
    fanout.append({
        "tabela": nome, "min": c.min(), "mediana": int(c.median()),
        "média": round(c.mean(), 2), "max": c.max(),
    })
pd.DataFrame(fanout)

,tabela,min,mediana,média,max
0,event_type,1,2,1.69,9
1,resource_type,1,1,1.15,5
2,log_feature,1,2,3.23,19


Um incidente pode ter até 9 tipos de evento, 5 tipos de recurso e 19 features de log.

É por isso que a modelagem do banco vai usar **tabelas-ponte** (uma linha por par
incidente–categoria) em vez de tentar espremer tudo em colunas. Uma tabela larga
precisaria de ~390 colunas e tornaria qualquer consulta SQL trivial e inútil.

---

## 10. Limitações do dataset

Registrar o que o dado **não** permite é tão importante quanto registrar o que permite.

**1. Não há dimensão temporal.** Nenhum dos sete arquivos possui coluna de data ou
hora. O `id` não tem semântica de ordem documentada. Portanto: sem séries temporais,
sem sazonalidade, sem MTTR, sem tendência. Essa limitação é do dataset, e este projeto
não vai simular tempo a partir do `id`.

**2. As categorias são anonimizadas.** `event_type 11`, `resource_type 8`,
`location 821` — não sabemos o que representam no mundo real. Podemos dizer que
o `resource_type 8` aparece em mais incidentes, mas não *por quê*. As conclusões
ficam no nível de padrão estatístico, não de causa física.

**3. 60% dos incidentes não têm rótulo.** Os 11.171 de `test.csv` foram descartados
por não terem `fault_severity`.

**4. Não há dados de duração, custo ou clientes afetados.** Não é possível
priorizar por impacto financeiro ou por número de assinantes.

---

## 11. Perguntas de negócio que este dataset responde

Com base em tudo acima, estas são as análises **viáveis** — e todas serão
implementadas em SQL e Python nos próximos checkpoints:

1. Como se distribuem os incidentes por gravidade?
2. Quais localidades concentram mais incidentes, em volume absoluto?
3. Quais localidades têm maior **taxa** de incidentes graves, com corte mínimo de amostra?
4. Quais tipos de evento são mais frequentes?
5. Quais tipos de evento estão associados a maior gravidade?
6. Quais recursos aparecem em mais incidentes, e quais concentram gravidade?
7. O `severity_type` do log prediz a gravidade real? (evidência forte encontrada acima)
8. O volume de log de um incidente se relaciona com sua gravidade?

**Descartadas por limitação do dado:** evolução temporal, sazonalidade, tempo de
reparo, impacto financeiro e qualquer interpretação causal das categorias.